# Lab 05: The $1 Pretraining Run

**Act IV — How They Learn** | GPU recommended | ~90–120 minutes

---

### What you will learn

1. **Build** a 10.7M-parameter GPT-2 model from scratch — and understand
   where each of those 10.7 million parameters lives
2. **Train** it on TinyStories, a dataset of synthetic children's stories
   used in real language model research (Eldan & Li, 2023)
3. **Watch** the loss curve descend from random noise (loss ~9.2) to
   coherent language (loss ~2.5) in minutes
4. **Test the textbook hypothesis**: train a second model on corrupted
   data and see, from your own experiment, that data quality beats
   data quantity
5. **Generate** text from both models and compare the results

---

### The idea

Everyone talks about pretraining, but very few people have actually done it.
Today you will. Not on a 7B model with a cluster — on a tiny model with
your laptop. The architecture is real (GPT-2 style), the data is real
(TinyStories), and the training dynamics are real (the loss curve, the
learning rate schedule, the generation quality). The only thing that is
small is the scale.

**Cost estimate:** This lab trains ~32M tokens on a 10.7M model. On a
Colab T4 that is roughly $0.10 of compute. On your laptop it is free.
The "$1" in the title is generous — you could run this 10 times over.

---
## 1. Setup

In [ ]:
import subprocess
import sys

try:
    import microscale
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/iqbal-sk/Microscale-labs.git",
        ]
    )
    # Force Python to see the newly installed package (important on Colab)
    import importlib

    importlib.invalidate_caches()
    import microscale

from microscale import apply_style, device_summary, get_torch_device, is_ci, show

apply_style()
device = get_torch_device()
print(device_summary())

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from rich.console import Console
from rich.table import Table

console = Console()

---
## 2. Build the Model

Our model is a standard GPT-2 architecture: token embeddings + position
embeddings, a stack of transformer blocks, then a language model head.

| Config | Value | Why |
|--------|-------|-----|
| Vocabulary | 10,000 tokens | Keeps embedding table small |
| Hidden dim | 320 | Narrow but deep |
| Heads | 5 | 64 dim per head |
| Layers | 6 | Enough depth for basic patterns |
| FFN dim | 1,280 | Standard 4x expansion |
| Context | 512 tokens | Fits a full TinyStory |
| Tied weights | Yes | Output head reuses embeddings |

In [ ]:
from microscale.tiny_gpt import TinyGPT

VOCAB_SIZE = 10000
SEQ_LEN = 512

model = TinyGPT(
    vocab_size=VOCAB_SIZE,
    d_model=320,
    n_heads=5,
    n_layers=6,
    d_ff=1280,
    max_seq_len=SEQ_LEN,
    tie_weights=True,
).to(device)

n_params = model.count_parameters()
console.print(f"\n  Model: [bold]{n_params:,}[/bold] trainable parameters")

# Parameter breakdown
table = Table(title="Parameter Budget")
table.add_column("Component", style="bold")
table.add_column("Parameters", justify="right")
table.add_column("Fraction", justify="right")

components = {
    "Token embeddings": model.tok_emb.weight.numel(),
    "Position embeddings": model.pos_emb.weight.numel(),
    "Attention (all layers)": sum(
        p.numel() for block in model.blocks for p in block.attn.parameters()
    ),
    "FFN (all layers)": sum(p.numel() for block in model.blocks for p in block.ffn.parameters()),
    "LayerNorms": sum(
        p.numel() for block in model.blocks for name, p in block.named_parameters() if "ln" in name
    )
    + model.ln_f.weight.numel()
    + model.ln_f.bias.numel(),
    "Output head": 0,  # tied with embeddings
}

for comp, count in components.items():
    table.add_row(comp, f"{count:,}", f"{count / n_params:.1%}")
console.print(table)

---
## 3. Prepare the Data

**TinyStories** is a dataset of ~2.1 million short children's stories
generated by GPT-3.5 and GPT-4 (Eldan & Li, 2023, Microsoft Research).
Stories use simple vocabulary that a 3-4 year old would understand, but
have correct grammar, coherent plots, and consistent characters.

We use the GPT-2 tokenizer but restrict to the 10,000 most common
tokens. Tokens outside this set are skipped during tokenization,
keeping only in-vocabulary tokens. This follows the approach from
the original TinyStories paper.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

print("Loading TinyStories dataset...")
# Load a concrete slice (not streaming) so cells can be re-run
N_STORIES = 5000 if is_ci() else 50000
raw_dataset = load_dataset("roneneldan/TinyStories", split=f"train[:{N_STORIES}]")

base_tokenizer = AutoTokenizer.from_pretrained("gpt2")
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
# We use GPT-2 tokenizer only for encoding, not for running GPT-2.
# Suppress the "sequence length > 1024" warning — irrelevant here.
base_tokenizer.model_max_length = 100000

console.print(f"  Loaded {len(raw_dataset):,} stories")

We need to convert stories into fixed-length token sequences. Tokens
outside our 10K vocabulary are dropped (not replaced with a dummy token,
which would pollute the training signal).

In [ ]:


def tokenize_and_chunk(stories, vocab_size: int, seq_len: int):
    """Tokenize stories, drop OOV tokens, pack into fixed-length chunks."""
    all_tokens = []
    for example in stories:
        tokens = base_tokenizer.encode(example["text"])
        # Keep only in-vocabulary tokens (drop the rest)
        tokens = [t for t in tokens if t < vocab_size]
        all_tokens.extend(tokens)

    total = len(all_tokens)
    # Trim to exact multiple of seq_len
    all_tokens = all_tokens[: (total // seq_len) * seq_len]
    sequences = torch.tensor(all_tokens, dtype=torch.long).view(-1, seq_len)
    return sequences


clean_data = tokenize_and_chunk(raw_dataset, VOCAB_SIZE, SEQ_LEN)
total_tokens = clean_data.numel()
console.print(
    f"  Tokenized: {total_tokens:,} tokens → "
    f"[bold]{clean_data.shape[0]:,}[/bold] sequences of {SEQ_LEN}"
)

---
## 4. Create Corrupted Data

To test the textbook hypothesis, we create a corrupted version of the
same data. Same number of tokens, same vocabulary — but the quality
is deliberately degraded:

- **Shuffled tokens**: Randomly reorder tokens within 30% of sequences,
  destroying sentence structure and coherence
- **Duplicated content**: Copy 30% of sequences over unique ones,
  reducing data diversity
- **Random noise**: Replace 5% of tokens with random vocabulary items

This simulates the gap between curated data and noisy web-crawled data.

In [ ]:


def corrupt_data(sequences: torch.Tensor, seed: int = 42) -> torch.Tensor:
    """Create a corrupted version of the training data."""
    rng = np.random.RandomState(seed)
    corrupted = sequences.clone()
    n_seqs = corrupted.shape[0]
    seq_len = corrupted.shape[1]

    # 1. Shuffle token order within 30% of sequences
    shuffle_idx = np.where(rng.random(n_seqs) < 0.3)[0]
    for i in shuffle_idx:
        corrupted[i] = corrupted[i][torch.randperm(seq_len)]

    # 2. Duplicate 30% of sequences (displaces unique content)
    n_dup = int(n_seqs * 0.3)
    src = rng.choice(n_seqs, n_dup)
    tgt = rng.choice(n_seqs, n_dup, replace=False)
    for s, t in zip(src, tgt):
        corrupted[t] = corrupted[s]

    # 3. Replace 5% of tokens with random tokens
    noise_mask = torch.tensor(rng.random(corrupted.shape) < 0.05)
    random_tokens = torch.randint(0, VOCAB_SIZE, corrupted.shape)
    corrupted[noise_mask] = random_tokens[noise_mask]

    return corrupted


corrupted_data = corrupt_data(clean_data)
console.print(f"  Corrupted dataset: {corrupted_data.shape[0]:,} sequences")

---
## 5. Training

We train two identical models — same architecture, same hyperparameters,
same number of steps — on different data.

Hyperparameters follow standard practice for small LM pretraining:

| Setting | Value | Notes |
|---------|-------|-------|
| Optimizer | AdamW | Standard for transformers |
| Learning rate | 5e-4 | High for small models |
| LR schedule | Cosine with warmup | Used by GPT-2, LLaMA, etc. |
| Beta2 | 0.95 | Lower than default 0.999, standard for LM |
| Weight decay | 0.1 | Regularization |
| Gradient clipping | 1.0 | Prevents exploding gradients |

In [ ]:
BATCH_SIZE = 32 if not is_ci() else 8
N_STEPS = 2000 if not is_ci() else 100
LR = 5e-4
WARMUP_STEPS = 200 if not is_ci() else 10


def train_model(name: str, data: torch.Tensor) -> tuple[TinyGPT, list[float]]:
    """Train a TinyGPT model and return (model, loss_history)."""
    mdl = TinyGPT(
        vocab_size=VOCAB_SIZE,
        d_model=320,
        n_heads=5,
        n_layers=6,
        d_ff=1280,
        max_seq_len=SEQ_LEN,
    ).to(device)

    optimizer = torch.optim.AdamW(
        mdl.parameters(),
        lr=LR,
        betas=(0.9, 0.95),
        weight_decay=0.1,
    )

    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return step / max(WARMUP_STEPS, 1)
        progress = (step - WARMUP_STEPS) / max(N_STEPS - WARMUP_STEPS, 1)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    n_seqs = data.shape[0]
    losses = []
    lr_history = []
    mdl.train()
    t0 = time.time()
    log_interval = max(1, N_STEPS // 10)
    tokens_per_step = BATCH_SIZE * SEQ_LEN

    console.print(
        f"  [dim]Config: {N_STEPS} steps, batch={BATCH_SIZE}, seq_len={SEQ_LEN}, lr={LR}[/dim]"
    )
    console.print(f"  [dim]Total tokens: {N_STEPS * tokens_per_step:,}[/dim]\n")

    for step in range(N_STEPS):
        idx = torch.randint(0, n_seqs, (BATCH_SIZE,))
        batch = data[idx].to(device)

        output = mdl(batch, labels=batch)
        loss = output["loss"]

        optimizer.zero_grad()
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0).item()
        optimizer.step()
        scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]
        losses.append(loss.item())
        lr_history.append(current_lr)

        # Log every 10% of training
        if (step + 1) % log_interval == 0 or step == 0:
            elapsed_so_far = time.time() - t0
            tok_per_sec = (step + 1) * tokens_per_step / elapsed_so_far
            avg_loss = np.mean(losses[-log_interval:])
            console.print(
                f"  Step {step + 1:5d}/{N_STEPS}"
                f"  loss={avg_loss:.4f}"
                f"  lr={current_lr:.2e}"
                f"  grad_norm={grad_norm:.2f}"
                f"  tok/s={tok_per_sec:,.0f}"
                f"  [{elapsed_so_far:.0f}s]"
            )

    elapsed = time.time() - t0
    tok_seen = N_STEPS * tokens_per_step
    avg_final = np.mean(losses[-50:])
    console.print(
        f"\n  [bold]{name} done:[/bold] {elapsed:.0f}s |"
        f" {tok_seen:,} tokens |"
        f" {tok_seen / elapsed:,.0f} tok/s |"
        f" final loss: [bold]{avg_final:.3f}[/bold]"
    )
    return mdl, losses

### Train Model A: Clean Data

In [ ]:
console.print("\n[bold]Training Model A on clean TinyStories...[/bold]\n")
model_clean, losses_clean = train_model("Clean", clean_data)

### Train Model B: Corrupted Data

In [ ]:
console.print("\n[bold]Training Model B on corrupted data...[/bold]\n")
model_corrupt, losses_corrupt = train_model("Corrupted", corrupted_data)

---
## 6. Compare the Loss Curves

This is where the textbook hypothesis becomes visible. Same architecture,
same compute, same number of tokens — the only variable is data quality.

In [ ]:
window = max(1, len(losses_clean) // 50)


def smooth(vals, w):
    return np.convolve(vals, np.ones(w) / w, mode="valid")


fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Loss curves
ax = axes[0]
ax.plot(losses_clean, alpha=0.12, color="#4a7c74")
ax.plot(losses_corrupt, alpha=0.12, color="#8b3a3a")
s_clean = smooth(losses_clean, window)
s_corrupt = smooth(losses_corrupt, window)
ax.plot(range(len(s_clean)), s_clean, color="#4a7c74", linewidth=2.5, label="Clean data")
ax.plot(range(len(s_corrupt)), s_corrupt, color="#8b3a3a", linewidth=2.5, label="Corrupted data")
ax.set_xlabel("Training Step")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Loss Curves: Clean vs Corrupted Data")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Final loss bars
ax = axes[1]
final_clean = float(np.mean(losses_clean[-50:]))
final_corrupt = float(np.mean(losses_corrupt[-50:]))
bars = ax.bar(
    ["Clean\nData", "Corrupted\nData"],
    [final_clean, final_corrupt],
    color=["#4a7c74", "#8b3a3a"],
    width=0.5,
    edgecolor="#1a1f3a",
    linewidth=1.5,
)
for bar, val in zip(bars, [final_clean, final_corrupt]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{val:.2f}",
        ha="center",
        fontsize=14,
        fontweight="bold",
    )
ax.set_ylabel("Final Loss (lower = better predictions)")
ax.set_title("Final Loss Comparison")
ax.set_ylim(0, max(final_clean, final_corrupt) * 1.3)

fig.suptitle(
    "The Textbook Hypothesis in Miniature",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
fig.tight_layout()
show(fig, filename="05-loss-curves.png")

In [ ]:
gap = final_corrupt - final_clean
pct = (gap / final_clean) * 100
console.print(f"\n  Clean final loss:     [bold green]{final_clean:.3f}[/]")
console.print(f"  Corrupted final loss: [bold red]{final_corrupt:.3f}[/]")
console.print(f"  Gap: [bold]+{gap:.3f}[/bold] ({pct:.0f}% higher loss on corrupted data)")

Same model, same compute budget, same token count. The only difference
is data quality — and it shows clearly in the loss curve. Curated data
trains more efficiently than noisy data. This is the core insight behind
Microsoft's Phi series of models.

---
## 7. Generate Text

Let's see what these models actually produce. We give both models the
same prompt and compare completions.

In [ ]:
PROMPTS = [
    "Once upon a time, there was a little",
    "The dog ran to the park and",
    "She was very happy because her",
    "One day, a boy found a big",
]


def generate_text(mdl, prompt, max_tokens=80):
    """Generate text from a prompt using our trained model."""
    mdl.eval()
    tokens = base_tokenizer.encode(prompt)
    # Keep only in-vocabulary tokens
    tokens = [t for t in tokens if t < VOCAB_SIZE]
    if not tokens:
        tokens = [base_tokenizer.eos_token_id]
    input_ids = torch.tensor([tokens], device=device)

    with torch.no_grad():
        output_ids = mdl.generate(
            input_ids,
            max_new_tokens=max_tokens,
            temperature=0.8,
            top_k=50,
        )

    return base_tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=True)

In [ ]:
console.print("\n[bold]Generation Comparison[/bold]\n")

for prompt in PROMPTS:
    console.print(f'  [bold]Prompt:[/bold] "{prompt}"\n')

    clean_text = generate_text(model_clean, prompt)
    corrupt_text = generate_text(model_corrupt, prompt)

    console.print("  [green]Clean model:[/green]")
    # Show only the generated part (after prompt)
    console.print(f"    {clean_text[:250]}\n")
    console.print("  [red]Corrupted model:[/red]")
    console.print(f"    {corrupt_text[:250]}\n")
    console.print("  " + "─" * 60 + "\n")

With enough training steps (2000+), the clean-data model produces
recognizable story fragments — simple sentences about characters doing
things. The corrupted-data model generates less coherent text because
it learned from shuffled, duplicated, and noisy sequences.

If you only ran 100 steps (CI mode), both models will produce mostly
noise — they need more training. Run with the full 2000 steps to see
the difference clearly.

---
## 8. Training Dynamics

Let's visualize the learning rate schedule and how the loss distribution
shifts during training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# LR schedule
ax = axes[0]
lr_vals = []
for step in range(N_STEPS):
    if step < WARMUP_STEPS:
        lr_val = LR * step / max(WARMUP_STEPS, 1)
    else:
        prog = (step - WARMUP_STEPS) / max(N_STEPS - WARMUP_STEPS, 1)
        lr_val = LR * 0.5 * (1 + np.cos(np.pi * prog))
    lr_vals.append(lr_val)

ax.plot(lr_vals, color="#b87333", linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Learning Rate")
ax.set_title("Cosine LR Schedule with Linear Warmup")
ax.axvline(
    WARMUP_STEPS,
    color="#6b7091",
    linestyle="--",
    alpha=0.5,
    label=f"Warmup ends (step {WARMUP_STEPS})",
)
ax.legend()
ax.grid(True, alpha=0.3)

# Loss distribution shift
ax = axes[1]
n_hist = min(200, len(losses_clean) // 4)
early = losses_clean[:n_hist]
late = losses_clean[-n_hist:]
ax.hist(early, bins=30, alpha=0.6, color="#8b3a3a", label=f"First {n_hist} steps", density=True)
ax.hist(late, bins=30, alpha=0.6, color="#4a7c74", label=f"Last {n_hist} steps", density=True)
ax.set_xlabel("Loss Value")
ax.set_ylabel("Density")
ax.set_title("Loss Distribution: Early vs Late Training")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("Training Dynamics", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
show(fig, filename="05-training-dynamics.png")

---
## 9. Cost Estimate and Saved Artifacts

Let's calculate the actual compute cost of this experiment.

In [ ]:
from microscale.viz import _output_dir

tok_trained = N_STEPS * BATCH_SIZE * SEQ_LEN * 2  # x2 for both models
flops_per_token = 6 * n_params  # ~6N for forward + backward
total_flops = tok_trained * flops_per_token

table = Table(title="Training Summary")
table.add_column("Metric", style="bold")
table.add_column("Value", justify="right")
table.add_row("Model size", f"{n_params:,} parameters")
table.add_row("Tokens trained (both models)", f"{tok_trained:,}")
table.add_row("Total FLOPs", f"{total_flops:.2e}")
table.add_row("Estimated T4 cost", f"~${total_flops / 6.5e13:.2f}")
table.add_row("Estimated M-series cost", "Free (your laptop)")
console.print(table)

# Save models
output_path = _output_dir()
torch.save(model_clean.state_dict(), output_path / "05-model-clean.pt")
torch.save(model_corrupt.state_dict(), output_path / "05-model-corrupted.pt")

clean_size = os.path.getsize(output_path / "05-model-clean.pt") / 1e6
console.print(f"\n  Models saved to outputs/ ({clean_size:.1f} MB each)")

---
## What You Learned

| Finding | Your evidence |
|---|---|
| Initial loss = ln(vocab_size) | Started at ~9.2 = ln(10000) |
| Clean data trains faster | Lower loss at same step count |
| Data quality > data quantity | Same tokens, different quality, different result |
| Cosine LR + warmup is standard | Smooth decay after linear ramp-up |
| Small models learn fast | 10M params, minutes on a laptop |

### Artifacts in `outputs/`

| File | What it is |
|------|-----------|
| `05-loss-curves.png` | Clean vs corrupted training curves |
| `05-training-dynamics.png` | LR schedule and loss distribution |
| `05-model-clean.pt` | Your trained clean-data model (43 MB) |
| `05-model-corrupted.pt` | Your trained corrupted-data model |

### References

- Eldan & Li, "TinyStories" (2023, arXiv:2305.07759)
- Gunasekar et al., "Textbooks Are All You Need" (2023, arXiv:2306.11644)
- Radford et al., "Language Models are Unsupervised Multitask Learners"
  (GPT-2, 2019)